## Extracción de información de una base de datos

In [1]:
import pandas as pd
from itertools import combinations, product

In [2]:
#Convertimos el archivo.txt a .csv
read_file = pd.read_csv ('EXsept2024.txt', delimiter = '\t')
read_file.to_csv ('EXsept2024.csv', index=False, sep = ';')

In [3]:
#Hacemos el dataframe
daf = pd.read_csv ('EXsept2024.csv', sep = ';')
atr =["a1","a2","a3","a4","a5","a6","a7"]  
daf.insert(0, "", atr, True) 
#daf.to_csv('ContextoLimpio.csv', index=False, sep=';')
daf = daf.drop(daf.columns[daf.shape[1]-1], axis=1)
daf #index=filas columns=columnas

,,Obj 1,Obj 2,Obj 3,Obj 4,Obj 5,Obj 6
0,a1,1,1,0,0,0,1
1,a2,0,0,1,1,1,1
2,a3,1,1,1,0,0,0
3,a4,0,0,0,1,1,1
4,a5,0,0,0,0,1,1
5,a6,1,1,0,1,1,1
6,a7,1,1,0,0,1,0


In [4]:
def extension(df, Y):
    if len(Y) == 0:  # La extensión del vacío es B (todas las columnas excepto la primera)
        return df.columns[1:].tolist()

    # Filtrar filas donde la primera columna (atributos) coincida con alguno en Y
    rows = df[df.iloc[:, 0].isin(Y)]

    # Obtener las columnas donde todas las filas tienen un 1 en esas columnas
    extension_result = rows.iloc[:, 1:].apply(lambda col: (col == 1).all(), axis=0)

    # Retornar las columnas donde el resultado es True (es decir, donde todos son 1)
    return extension_result[extension_result].index.tolist()

def intension(df, X):
    if len(X) == 0:  # La intension del vacío es A (todas las filas)
        return df.iloc[:, 0].tolist()

    # Filtrar las columnas que están en X
    columns = df.columns.isin(X)
    filtered_df = df.iloc[:, columns]

    # Filtrar filas donde todas las columnas tienen valor 1
    rows = df[(filtered_df == 1).all(axis=1)]

    # Retornar los valores de la primera columna de las filas que cumplen la condición
    return rows.iloc[:, 0].tolist()

In [5]:
def clausura(df, lista):
    # Definir si la lista es de columnas (atributos) o filas (objetos)
    if lista[0] in df.columns:
        objetos = extension(df, intension(df, lista))
        atributos = intension(df, objetos)
    else:
        atributos = intension(df, extension(df, lista))
        objetos = intension(df, atributos)

    return [objetos, atributos]

def comprobar(df, lista):
    # Obtener la clausura de la lista
    clausura_result = clausura(df, lista)

    # Comprobar si la lista es un concepto (si coincide con su clausura)
    if clausura_result == lista:
        print("La lista SÍ es un concepto.")
        return True
    else:
        print("La lista NO es un concepto porque su clausura es ",clausura_result)
        return False

In [6]:
def tablaconceptos(df):
    # Inicializamos la lista de conceptos
    conceptos = []  
    B=extension(df,[])
    
    # Lista para contar los unos (1s) por fila, sin la columna de nombres
    fila = df.iloc[:, 1:].sum(axis=1).tolist()  # Sumamos por fila excluyendo la primera columna
    #print(fila)
   
    #Dejo guardadas las posiciones de los atributos que no tengan relación con ningún objeto 
    posiciones_ceros = [index for index, valor_fila in enumerate(fila) if valor_fila == 0]
    
    # Recorremos las filas (atributos) para verificar si alguna tiene todos los valores a 1
    atributos_todos_unos = []
    for index, row in df.iterrows():
    # Verificamos si todos los valores en la fila son iguales a 1
        if all(value == 1 for value in row[1:]):  # Excluyendo la primera columna (nombres de los objetos)
            atributos_todos_unos.append(row[0])
            fila[index] = 0 #lo quitamos de la lista 
    
    if len(atributos_todos_unos)>0:
        # Si existe al menos un atributo con todos sus valores a 1, el primer concepto es el conjunto total de objetos
        conceptos.append([atributos_todos_unos, B])
    else:
        # Si no existe, el primer concepto es el vacío
        conceptos.append([[], B])
    
    # Si todas las filas están vacías (no tienen 1s), el único concepto es el inicial
    if max(fila) == 0:
        return conceptos

    # Ahora seguimos con el resto del algoritmo para calcular los demás conceptos
    while max(fila) > 0:
        # Identificamos la fila con más unos
        posicion_max = fila.index(max(fila))
        namefila = df.iloc[posicion_max, 0]  # Obtenemos el nombre del objeto
        #print("Por orden maxdefila, posicion y nombre:",max(fila),posicion_max,namefila)
        # Buscamos la extensión del nombre de la fila actual
        ext_actual = extension(df, [namefila])

        # Buscamos si la extensión ya está en la tabla de conceptos
        encontrada = False
        for i, concepto in enumerate(conceptos):
            if set(concepto[1]) == set(ext_actual):  # Si ya está la extensión
                conceptos[i][0].append(namefila)  # Añadimos el atributo a su concepto
                encontrada = True
                break
        
        if not encontrada:
            # Si no está la extensión en la tabla, añadimos un nuevo concepto
            conceptos.append([[namefila], ext_actual])

            # Calculamos las intersecciones con conceptos anteriores
            for i in range(len(conceptos) - 1):  # Hasta el concepto recién añadido
                interseccion = list(set(conceptos[i][1]) & set(ext_actual))  # Intersección con cada concepto anterior

                # Vemos si la intersección ya está o no en la tabla
                if not any(set(concepto[1]) == set(interseccion) for concepto in conceptos):
                    conceptos.append([[], interseccion,'intersección de conceptos'])  # Añadimos la intersección si no está

        # Marcamos la fila como procesada
        fila[posicion_max] = 0

    #Finalmente, si hemos tenido atributos sin ninguna relación, lo añadimos al concepto que va con el vacío   
    for i in range(len(conceptos)):
        if conceptos[i][1] == []:  # Verifica si la segunda posición es una lista vacía
            for j in range(len(posiciones_ceros)):
                namefila = df.iloc[posiciones_ceros[j], 0]
                conceptos[i][0].append(namefila)  # Añade el nombre de los atributos sin relación a la primera posición

    return conceptos

In [7]:
tablaconceptos(daf)

[[[], ['Obj 1', 'Obj 2', 'Obj 3', 'Obj 4', 'Obj 5', 'Obj 6']],
 [['a6'], ['Obj 1', 'Obj 2', 'Obj 4', 'Obj 5', 'Obj 6']],
 [['a2'], ['Obj 3', 'Obj 4', 'Obj 5', 'Obj 6']],
 [['a4'], ['Obj 6', 'Obj 4', 'Obj 5'], 'intersección de conceptos'],
 [['a1'], ['Obj 1', 'Obj 2', 'Obj 6']],
 [[], ['Obj 6'], 'intersección de conceptos'],
 [['a3'], ['Obj 1', 'Obj 2', 'Obj 3']],
 [[], ['Obj 1', 'Obj 2'], 'intersección de conceptos'],
 [[], ['Obj 3'], 'intersección de conceptos'],
 [[], [], 'intersección de conceptos'],
 [['a7'], ['Obj 1', 'Obj 2', 'Obj 5']],
 [[], ['Obj 5'], 'intersección de conceptos'],
 [['a5'], ['Obj 5', 'Obj 6']]]

In [8]:
def extensiones(df):
    
    # Calculamos la extensión para el conjunto vacío (esto sería el universo completo)
    B = extension(df, [])
    
    # Obtenemos los nombres de los atributos
    atr = intension(df,[])

    #Creamos una lista que guardará todas las extensiones    
    all_extensiones = []
    
    # Calculamos la extensión para cada atributo
    for i in range(len(atr)):
        atributo_actual = atr[i]
        ext_actual = extension(df, [atributo_actual])
        all_extensiones.append(ext_actual) # Guardamos todas las extensiones de cada atributo
        print(f"{atributo_actual}' = {ext_actual}")
    return all_extensiones

In [9]:
def clasificar(df):
    
    # Inicialización de las listas de clasificación
    infimo_irreducibles = []  # Mf: infimos irreducibles
    infimo_reducibles = []    # Infimos reducibles
    necesario = []            # Cf: absolutamente necesario
    absolutamente_innecesario = []  # If: absolutamente innecesario
    relativ_necesario = []    # Kf: relativamente necesario
    
    # Calculamos la extensión para el conjunto vacío (esto sería el universo completo)
    B = extension(df, [])
    
    tab=tablaconceptos(df)
    
    #Si en el primer concepto cuya extensión es B hay un atibuto este será reducible e innecesario
    if(tab[0][0]!=[]): 
        for j in range(len(tab[0][0])):
            infimo_reducibles.append(tab[0][0][j])
            absolutamente_innecesario.append(tab[0][0][j])
    
    #Si hay un concepto con mas de un atributo estos serán relativamente necesarios e irreducibles
    for i in range(1,len(tab)):
        if(len(tab[i][0])>1):
            if(len(tab[i])==3): #si es intersección son inncesarios y reducibles
                for j in range(len(tab[i][0])):
                    infimo_reducibles.append(tab[i][0][j])
                    absolutamente_innecesario.append(tab[i][0][j])
            else:
                for j in range(len(tab[i][0])):
                    infimo_irreducibles.append(tab[i][0][j])
                    relativ_necesario.append(tab[i][0][j])
            continue
            
        if(tab[i][0]!=[]):
            if(len(tab[i])==3): #es intersección
                for j in range(len(tab[i][0])):
                    infimo_reducibles.append(tab[i][0][j])
                    absolutamente_innecesario.append(tab[i][0][j])
            else:               #no es intersección
                for j in range(len(tab[i][0])):
                    infimo_irreducibles.append(tab[i][0][j])
                    necesario.append(tab[i][0][j])
            
    return {
        "Infimos irreducibles (M_f)": infimo_irreducibles,
        "Infimos reducibles": infimo_reducibles,
        "Absolutamente necesarios (C_f)": necesario,
        "Relativamente necesarios (K_f)": relativ_necesario,
        "Absolutamente innecesarios (I_f)": absolutamente_innecesario
    }

In [10]:
clasificar(daf)

{'Infimos irreducibles (M_f)': ['a6', 'a2', 'a1', 'a3', 'a7', 'a5'],
 'Infimos reducibles': ['a4'],
 'Absolutamente necesarios (C_f)': ['a6', 'a2', 'a1', 'a3', 'a7', 'a5'],
 'Relativamente necesarios (K_f)': [],
 'Absolutamente innecesarios (I_f)': ['a4']}

In [11]:
def reductosyconsistentes(df):
    
    # Clasificación de los atributos
    clasificacion = clasificar(df)
    C_f = clasificacion['Absolutamente necesarios (C_f)']
    K_f = clasificacion['Relativamente necesarios (K_f)']
    I_f = clasificacion['Absolutamente innecesarios (I_f)']

    if not C_f and not K_f and not I_f:
        print("No existe reducto ni conjunto consistente.")
        return

    if not K_f and not I_f:
        print("El único reducto es el formado por los C_f:", C_f)
        return

    # Agrupación de atributos relativamente necesarios según su extensión
    extension_grupos = {}
    for atributo in K_f:
        ext = tuple(sorted(extension(df, [atributo])))  # Ordenar para facilitar la comparación
        if ext in extension_grupos:
            extension_grupos[ext].append(atributo)
        else:
            extension_grupos[ext] = [atributo]

    # Generar combinaciones posibles de reductos
    grupos = list(extension_grupos.values())
    reductos_posibles = []

    # Obtener todas las combinaciones posibles seleccionando un atributo de cada grupo
    for seleccion in product(*grupos):
        reducto = C_f + list(seleccion)
        reductos_posibles.append(reducto)

    # Generar conjuntos consistentes: Reductos + combinaciones de I_f y K_f
    consistentes_posibles = []
    for r in reductos_posibles:
        for i_comb in range(len(I_f) + 1):
            for i_seleccion in combinations(I_f, i_comb):
                consistente = r + list(i_seleccion)
                consistentes_posibles.append(consistente)
        
    return {
        "Reductos": reductos_posibles,
        "Consistentes": consistentes_posibles
    }

In [12]:
#Comprobar si un conjunto de atributos es consistente o reducto
lista = ['a6', 'a2', 'a1', 'a3', 'a7', 'a5', 'a4']

if lista in reductosyconsistentes(daf)["Consistentes"]:
    print("Si es consistente")
else:
    print("No es consistente")
    
if lista in reductosyconsistentes(daf)["Reductos"]:
    print("Si es reducto")
else:
    print("No es reducto")

Si es consistente
No es reducto


In [13]:
def implicacion_valida(atr1, atr2, df):
    try:
        composicion = intension(df,extension(df,atr1))
        for x in atr2:
            if x not in composicion:
                return False
        return True
    except:
        print('ERROR')
        return

In [14]:
# Comprobamos las siguientes implicaciones:

print('1. {a1} -> {a4} es ', implicacion_valida(['a1'],['a4'], daf))
print('1. {a1} -> {a2} es ', implicacion_valida(['a1'],['a2'], daf))
print('1. {a2,a7} -> {a6} es ', implicacion_valida(['a2','a7'],['a6'], daf))

1. {a1} -> {a4} es  False
1. {a1} -> {a2} es  False
1. {a2,a7} -> {a6} es  True
